# Reading Eigensolution Results from ESCDF to Exodus

This document will demonstrate how to take in eigensolution results (i.e. Modal Data) in the ESCDF format and translate it to an Exodus file for visualization in Paraview or other such tools using Matlab (with the 1553 Exodus Utilities) or Python (with SDynPy).

## Preliminary Setup
To start, we will perform preliminary operations needed by our respective programming languages.  In Matlab, we clear and reset the workspace.  In Python, we import our modules.  In both cases, we need to ensure the respective libraries needed are on the Matlab or Python path.

In [1]:
% Reading Eigensolution Results from ESCDF to Exodus
% Example problem created by Dan Rohe, 1522

% Clear the workspace
clear; clc; close all;

% If you don't have exodus tools or ESCDF on your path,
% uncomment the following addpath functions and point
% them to the correct paths.

% addpath(genpath('/path/to/MatlabExodusUtilities1553'));
% addpath(genpath('/path/to/escdf'));

## Loading in ESCDF Results

The next step will be to load in the ESCDF files that we will be reading.  We will assume we have either created these files following the previous documentation page, or we have downloaded them from the above links.  To demonstrate the interoperability with ESCDF, we will open the version created using Matlab with Python and vice versa.

In [2]:
% Load in the Modal Results Data

% To show interoperability, we will load in the ESCDF file created by
% Python in this file.
esfile = escdf.load('framewing_eigen_escdf_py.esf');

If we type in `esfile` into the command window or console, we will see the familiar structure of our ESCDF file.

In [3]:
esfile

esfile = 
  escdf object
    Created by dprohe on 16-Apr-2026 15:47:00

  Metadata:

    attributes (global_analysis_attributes)
    mesh_geometry (geometry)

  Activities:
    eigen:
      Eigensolution of frame_wing_thick_v2_si-eig.exo
      Date: 2022-10-26 16:11:39.000
      Data:
        eigen
      Metadata Links:
        mesh_geometry, attributes

## Identifying the Relevant Data and Metadata

In general, ESCDF files may contain many activities, each which link to many sets of data and many pieces of metadata.  They may also have different reference names assigned to them.  Therefore, when writing general functions to read in arbitrary ESCDF files, one should always reference the relevant activity or data, and use the links in ESCDF to pull the correct metadata associated with that activity, using the type of data or metadata to perform selection, rather than the name which might change from file to file.

In this case, the activity we want is the first (and only) activity in the file.  A general function may take as its input argument the activity name or activity index.  Here we simply select the first index.

In [4]:
% Identify the modal data we are after

% We will identify the activity containing the modal data
esact = esfile.activities(1);

Querying the `esact` variable in the command window or console will show us information on the activity.

In [5]:
esact

esact = 
    eigen:
      Eigensolution of frame_wing_thick_v2_si-eig.exo
      Date: 2022-10-26 16:11:39.000
      Data:
        eigen
      Metadata Links:
        mesh_geometry, attributes

We will then use the links in the activity to select from the metadata objects associated with the activity.  We will select the metadata based on its type being `geometry`.

In [6]:
% A general ESCDF file may have multiple geometries and other metadata,
% so best practice is to use links between the activity and metadata
% to find the relevant geometry
metadata_links = esact.get_metadata_links();
% Find the metadata that is a geometry
esgeo = [];
for i = 1:length(metadata_links)
    metadata = esfile.metadata.(metadata_links{i});
    if metadata.istype('geometry')
        esgeo = metadata;
    end
end
if isempty(esgeo)
    error(['No geometry found linked to activity ',esactivity.get_name()])
end

We will do a similar process to select data in the activity that has the type `mode`.

In [7]:
% Get the data in a similar way
esmode = [];
esdata = esact.get_data();
for i = 1:length(esdata)
    data = esdata(i);
    if data.istype('mode')
        esmode = data;
    end
end
if isempty(esmode)
    error(['No mode data found in activity ',esactivity.get_name()])
end

## Extracting Geometry Information
We can remind ourselves of the properties in the geometry object by querying `esgeo` in the command window or console.  One thing to note is that all properties state `on disk`, meaning that ESCDF has not loaded them into memory.  Requesting data from these fields will read the data directly from disk.

In [8]:
esgeo

esgeo = 
  escdf_dataset: mesh_geometry (geometry v0.1.0)

  Properties:

      attachment_names: []
           attachments: []
         element_color: []
    element_connection: u8, (57316), on disk
          element_type: str, (57316), on disk
            line_color: []
       line_connection: []
               node_id: u8, (291977), on disk
         node_position: f8, (291977       3), on disk
      node_x_direction: f8, (291977       3), on disk
      node_y_direction: f8, (291977       3), on disk
      node_z_direction: f8, (291977       3), on disk
                 notes: []
        position_units: str, (), on disk

To assemble an Exodus file, we need to extract the data in the `esgeo` object to produce a node number map, a node coordinate array, and the element blocks.  We should also pull out the local coordinate system information.  While this file is built entirely in global coordinates, arbitrary modal data that one might receive in ESCDF format may have local coordinate information, so any general scripts we might write should automatically look for that information.

One thing to also note is that we have not stored any information that would allow us to reconstruct which elements originally belonged to which element block.  The way we stored this information was more for results visualization than for reconstructing the analysis.  Therefore, we will simply reconstruct one element block per element type in the Exodus file.

In [9]:
% Extract the geometry information

% Exodus needs node and element information to set up the exodus file.
% Let's extract that information
node_ids = esgeo.node_id(:);
coords = esgeo.node_position(:);

% We should also extract the local coordinate system information.
% Note that this particular file has everything in global coordinates,
% however, a general modal data package from an experimental group
% might not.  Therefore we should get used to understanding how to
% work with local coordinate system directions
node_x_dir = esgeo.node_x_direction(:);
node_y_dir = esgeo.node_y_direction(:);
node_z_dir = esgeo.node_z_direction(:);

% Finally, let's extract the element information.  We've lost all
% all references to the original element blocks.  Also, a general
% modal package from an experimental group would never have had
% blocks to begin with, so we should go through and create new
% blocks based on element types anyway
all_element_types = esgeo.element_type(:);
all_element_connectivities = esgeo.element_connection(:);

To construct element blocks, we will loop through the different element types and select rows of the connectivity matrix corresponding to that type of element.  These we can then assemble into a regular array (instead of the original ragged array).  Note that the connectivity is stored in ESCDF with reference to the node number, not the node index like Exodus.  Therefore, we must create an inverse node map that maps the node number back to the index.  Similarly, Exodus element block names are not identical to ESCDF, so we should create an element type map as well.

In [10]:
element_types = unique(all_element_types);

% We need to map the element types in ESCDF to the element types in
% Exodus
elem_map = containers.Map({'sphere1','bar2','hex20'},{'SPHERE','BEAM','HEX20'});
% We also need to map the node numbers back to the indices to use with
% Exodus
node_map_inv = containers.Map(node_ids,1:length(node_ids));

% Loop through each block type to extract the connectivity matrix
block_connectivities = {};
for i = 1:length(element_types)
    block_indices = strcmp(all_element_types,element_types{i});
    block_connectivity = [all_element_connectivities{block_indices,:}].';
    % Apply the inverse node map.  It seems that maps only accept one item
    % at a time, so we need to use arrayfun to map each item.
    block_connectivities{end+1} = arrayfun(@(item) node_map_inv(item),...
        block_connectivity);
end

% We should also map our element types, this time again using cellfun
exodus_block_types = cellfun(@(item) elem_map(item), element_types, ...
    'UniformOutput',false);

## Extracting Modal Information
We must similarly extract modal information.  We can query the `esmode` object by typing it into the command window or console to remind ourselves of its properties.

In [11]:
esmode

esmode = 
  escdf_dataset: eigen (mode v0.1.0)

  Properties:

    attachment_names: []
         attachments: []
       damping_ratio: []
         description: []
            dof_name: str, (876501), on disk
           frequency: f8, (80), on disk
          modal_mass: []
               notes: []
               shape: f8, (876501      80), on disk

We will extract the frequencies, the shape matrix, and the degree of freedom names from the dataset.

In [12]:
% Extract Modal Information

% With the geometry extracted, we can now pull the modal data from the
% file.  In general, the mode shape information cannot be assumed to be
% in the same order as the node id numbers in the geometry, so we will have
% to use the dof_name property to help us sort the data.  Additionally,
% we will use the local node directions to help us convert the data to
% the global displacements desired by the exodus file.

frequencies = esmode.frequency(:);
shape_array = esmode.shape(:);
dof_names = esmode.dof_name(:);

In Exodus, natural frequency information is stored as the time step.  Mode shape information is by default stored in variable names `DispX`, `DispY`, and `DispZ` for displacements and `RotX`, `RotY`, and `RotZ` for rotations.  We must create an array of values for each of these variables.  These will be nodal variables in the Exodus file, so there should be one value for each node for each time step.  We will assemble these as 3D arrays with shape `(num_nodes, 3, num_freqs)`.

In [13]:
% Initialize an array of zeros
displacement_shapes = zeros(length(node_ids),3,length(frequencies));
rotation_shapes = zeros(length(node_ids),3,length(frequencies));

ESCDF packages the degree of freedom names along with the shape matrix.  Therefore, we should never assume that the shape matrix is ordered in any specific way.  We must parse the degree of freedom names and reference it back to the node number map in order to identify the correct row of the nodal variable array we are creating.  Similarly, we should not assume that we are working in global coordinate systems and simply assign `X` degree of freedom values to the first column.  We must multiply `X` degrees of freedom by the `node_x_direction vector` to reconstruct the global motion.

We can use regular expressions to parse out the node number, direction, and polarity of the degree of freedom name.  The node number is passed to the node map to produce the node index, and these are used along with the direction string to select the direction vector.  The shape coefficient is then multiplied by the direction vector, and it is added to the proper displacement or rotation shape matrix.  By adding the values, we allow the contributions from the X, Y, and Z displacements to be summed together rather than the Z overwriting the X, for example.

In [14]:
% Go through each degree of freedom in the shape list and add it to the
% correct index
name_pattern = '(\d+)([a-zA-Z]+)([+-])';
name_parts = regexp(dof_names,name_pattern,'tokens');
for i = 1:length(dof_names)
    % Extract the parts of the dof name
    node_num = str2double(name_parts{i}{1}{1});
    direction_str = name_parts{i}{1}{2};
    polarity = str2double([name_parts{i}{1}{3},'1']);
    % Get the node index from the node map
    node_index = node_map_inv(node_num);
    % Use the direction string to select the correct direction vector
    if contains(direction_str,'X')
        direction = polarity*node_x_dir(node_index,:);
    elseif contains(direction_str,'Y')
        direction = polarity*node_y_dir(node_index,:);
    elseif contains(direction_str,'Z')
        direction = polarity*node_z_dir(node_index,:);
    else
        error('Unknown Direction')
    end
    shape_coefficients = shape_array(i,:);
    % Multiply the shape coefficients by the direction vector (using
    % broadcasting) and add to the correct matrix.  We add because there is
    % a contribution from X, Y, and Z local directions.
    if contains(direction_str,'R')
        rotation_shapes(node_index,:,:) = squeeze(rotation_shapes(node_index,:,:)) + direction.'*shape_coefficients;
    else
        displacement_shapes(node_index,:,:) = squeeze(displacement_shapes(node_index,:,:)) + direction.'*shape_coefficients;
    end
end

## Building the Exodus File
Now that we have constructed all of the data that the Exodus file might need, we can put it into the Exodus format.  This will look significantly different between Matlab and Python due to the differences in the 1553 Exodus Utilities and SDynPy Exodus implementation.  The 1553 Exodus Utilities construct the Exodus file in memory then write it to disk at the end.  The SDynPy Exodus tools create a file on disk immediately, then the various values are stored to that file via its function calls.

We can query the exodus object exo to see properties of the exodus file we've created.

In [16]:
% Build the exodus file

% Now we need to package the extracted data into the Exodus format.
% We will begin by creating an empty file
exo = FEMesh.Exodus();
% Add geometry information
exo.Nodes = FEMesh.Nodes(coords,node_ids,{'x','y','z'});
exo.Blocks = FEMesh.Blocks(1:length(exodus_block_types));
[exo.Blocks.ElementType] = exodus_block_types{:};
[exo.Blocks.Connectivity] = block_connectivities{:};
[exo.Blocks.Name] = exodus_block_types{:};
% Add modal information
exo.Time = frequencies;
exo.NodalVars = FEMesh.NodalVars({'DispX','DispY','DispZ','RotX','RotY','RotZ'});
exo.NodalVars(1).Data = squeeze(displacement_shapes(:,1,:));
exo.NodalVars(2).Data = squeeze(displacement_shapes(:,2,:));
exo.NodalVars(3).Data = squeeze(displacement_shapes(:,3,:));
exo.NodalVars(4).Data = squeeze(rotation_shapes(:,1,:));
exo.NodalVars(5).Data = squeeze(rotation_shapes(:,2,:));
exo.NodalVars(6).Data = squeeze(rotation_shapes(:,3,:));
% Make everything else empty
exo.Nodesets=FEMesh.Nodesets([]);
exo.Sidesets=FEMesh.Sidesets([]);
exo.GlobalVars=FEMesh.GlobalVars([]);
exo.ElemVars=FEMesh.ElemVars(1,[]);
exo.NodesetVars=FEMesh.NodesetVars([]);
exo.SidesetVars=FEMesh.SidesetVars([]);

% Query the Exodus File
exo

% Write the data to disk
exo_put('framewing_eigen_from_escdf_mat.exo',exo);

Database:  



Number of coordinates per node       =             3
Number of nodes                      =        291977
Number of elements                   =         57316
Number of element blocks             =             3

Number of nodal point sets           =             0
Number of element side sets          =             0
Number of global variables           =             0
Number of variables at each node     =             6
Number of variables at each element  =             0
Number of variables at each nodeset  =             0
Number of variables at each sideset  =             0

Number of time steps in the database =       80


We can load these models into a tool such as [Paraview](https://www.paraview.org/) to verify that the Exodus files were created successfully.

![image-2025-11-6_17-1-42.png](_attachments/sierrasd_eigen_read_from_escdf/cell016/6e716884-7601-428c-9f3e-acf1c50b37a4.png)